<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2025-Tutorial-Notebooks/blob/main/exercises/ex2/Exercise_2_Word_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
%matplotlib inline

In [86]:
# # run this cell only if you're working with Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

In [87]:
import torch

In [88]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# or for mac
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


### Source: [link](https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html#exercise-computing-word-embeddings-continuous-bag-of-words)

# Word Embeddings: Encoding Lexical Semantics

Word embeddings are dense vectors of real numbers, one per word in your
vocabulary. In NLP, it is almost always the case that your features are
words! But how should you represent a word in a computer? You could
store its ascii character representation, but that only tells you what
the word *is*, it doesn't say much about what it *means* (you might be
able to derive its part of speech from its affixes, or properties from
its capitalization, but not much). Even more, in what sense could you
combine these representations? We often want dense outputs from our
neural networks, where the inputs are $|V|$ dimensional, where
$V$ is our vocabulary, but often the outputs are only a few
dimensional (if we are only predicting a handful of labels, for
instance). How do we get from a massive dimensional space to a smaller
dimensional space?

How about instead of ascii representations, we use a one-hot encoding?
That is, we represent the word $w$ by

\begin{align}\overbrace{\left[ 0, 0, \dots, 1, \dots, 0, 0 \right]}^\text{|V| elements}\end{align}

where the 1 is in a location unique to $w$. Any other word will
have a 1 in some other location, and a 0 everywhere else.

There is an enormous drawback to this representation, besides just how
huge it is. It basically treats all words as independent entities with
no relation to each other. What we really want is some notion of
*similarity* between words. Why? Let's see an example.

Suppose we are building a language model. Suppose we have seen the
sentences

* The mathematician ran to the store.
* The physicist ran to the store.
* The mathematician solved the open problem.

in our training data. Now suppose we get a new sentence never before
seen in our training data:

* The physicist solved the open problem.

Our language model might do OK on this sentence, but wouldn't it be much
better if we could use the following two facts:

* We have seen  mathematician and physicist in the same role in a sentence. Somehow they
  have a semantic relation.
* We have seen mathematician in the same role  in this new unseen sentence
  as we are now seeing physicist.

and then infer that physicist is actually a good fit in the new unseen
sentence? This is what we mean by a notion of similarity: we mean
*semantic similarity*, not simply having similar orthographic
representations. It is a technique to combat the sparsity of linguistic
data, by connecting the dots between what we have seen and what we
haven't. This example of course relies on a fundamental linguistic
assumption: that words appearing in similar contexts are related to each
other semantically. This is called the `distributional
hypothesis <https://en.wikipedia.org/wiki/Distributional_semantics>`__.


# Getting Dense Word Embeddings

How can we solve this problem? That is, how could we actually encode
semantic similarity in words? Maybe we think up some semantic
attributes. For example, we see that both mathematicians and physicists
can run, so maybe we give these words a high score for the "is able to
run" semantic attribute. Think of some other attributes, and imagine
what you might score some common words on those attributes.

If each attribute is a dimension, then we might give each word a vector,
like this:

\begin{align}q_\text{mathematician} = \left[ \overbrace{2.3}^\text{can run},
   \overbrace{9.4}^\text{likes coffee}, \overbrace{-5.5}^\text{majored in Physics}, \dots \right]\end{align}

\begin{align}q_\text{physicist} = \left[ \overbrace{2.5}^\text{can run},
   \overbrace{9.1}^\text{likes coffee}, \overbrace{6.4}^\text{majored in Physics}, \dots \right]\end{align}

Then we can get a measure of similarity between these words by doing:

\begin{align}\text{Similarity}(\text{physicist}, \text{mathematician}) = q_\text{physicist} \cdot q_\text{mathematician}\end{align}

Although it is more common to normalize by the lengths:

\begin{align}\text{Similarity}(\text{physicist}, \text{mathematician}) = \frac{q_\text{physicist} \cdot q_\text{mathematician}}
   {\| q_\text{\physicist} \| \| q_\text{mathematician} \|} = \cos (\phi)\end{align}

Where $\phi$ is the angle between the two vectors. That way,
extremely similar words (words whose embeddings point in the same
direction) will have similarity 1. Extremely dissimilar words should
have similarity -1.


You can think of the sparse one-hot vectors from the beginning of this
section as a special case of these new vectors we have defined, where
each word basically has similarity 0, and we gave each word some unique
semantic attribute. These new vectors are *dense*, which is to say their
entries are (typically) non-zero.

But these new vectors are a big pain: you could think of thousands of
different semantic attributes that might be relevant to determining
similarity, and how on earth would you set the values of the different
attributes? Central to the idea of deep learning is that the neural
network learns representations of the features, rather than requiring
the programmer to design them herself. So why not just let the word
embeddings be parameters in our model, and then be updated during
training? This is exactly what we will do. We will have some *latent
semantic attributes* that the network can, in principle, learn. Note
that the word embeddings will probably not be interpretable. That is,
although with our hand-crafted vectors above we can see that
mathematicians and physicists are similar in that they both like coffee,
if we allow a neural network to learn the embeddings and see that both
mathematicians and physicists have a large value in the second
dimension, it is not clear what that means. They are similar in some
latent semantic dimension, but this probably has no interpretation to
us.


In summary, **word embeddings are a representation of the *semantics* of
a word, efficiently encoding semantic information that might be relevant
to the task at hand**. You can embed other things too: part of speech
tags, parse trees, anything! The idea of feature embeddings is central
to the field.


# Word Embeddings in Pytorch

Before we get to a worked example and an exercise, a few quick notes
about how to use embeddings in Pytorch and in deep learning programming
in general. Similar to how we defined a unique index for each word when
making one-hot vectors, we also need to define an index for each word
when using embeddings. These will be keys into a lookup table. That is,
embeddings are stored as a $|V| \times D$ matrix, where $D$
is the dimensionality of the embeddings, such that the word assigned
index $i$ has its embedding stored in the $i$'th row of the
matrix. In all of my code, the mapping from words to indices is a
dictionary named word\_to\_ix.

The module that allows you to use embeddings is torch.nn.Embedding,
which takes two arguments: the vocabulary size, and the dimensionality
of the embeddings.

To index into this table, you must use torch.LongTensor (since the
indices are integers, not floats).




In [89]:
# Author: Robert Guthrie

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
torch.manual_seed(1)

In [90]:
word_to_ix = {"hello": 0, "world": 1}
embeds = nn.Embedding(2, 5)  # 2 words in vocab, 5 dimensional embeddings
lookup_tensor = torch.tensor([word_to_ix["hello"]], dtype=torch.long)
hello_embed = embeds(lookup_tensor)
print(hello_embed)

tensor([[ 0.6614,  0.2669,  0.0617,  0.6213, -0.4519]],
       grad_fn=<EmbeddingBackward0>)


# An Example: N-Gram Language Modeling

Recall that in an n-gram language model, given a sequence of words
$w$, we want to compute

\begin{align}P(w_i | w_{i-1}, w_{i-2}, \dots, w_{i-n+1} )\end{align}

Where $w_i$ is the ith word of the sequence.

In this example, we will compute the loss function on some training
examples and update the parameters with backpropagation.

In [91]:
CONTEXT_SIZE = 2
EMBEDDING_DIM = 10
# We will use Shakespeare Sonnet 2
test_sentence = """When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a totter'd weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say, within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserv'd thy beauty's use,
If thou couldst answer 'This fair child of mine
Shall sum my count, and make my old excuse,'
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.""".split()
# we should tokenize the input, but we will ignore that for now
# build a list of tuples.  Each tuple is ([ word_i-2, word_i-1 ], target word)
trigrams = [([test_sentence[i], test_sentence[i + 1]], test_sentence[i + 2])
            for i in range(len(test_sentence) - 2)]
# print the first 3, just so you can see what they look like
print(trigrams[:3])

vocab = set(test_sentence)
word_to_ix = {word: i for i, word in enumerate(vocab)}


class NGramLanguageModeler(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs


losses = []
loss_function = nn.NLLLoss() # Negative Log Likelihood Loss
model = NGramLanguageModeler(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

for epoch in range(10):
    total_loss = 0
    for context, target in trigrams:

        # Step 1. Prepare the inputs to be passed to the model (i.e, turn the words
        # into integer indices and wrap them in tensors)
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)

        # Step 2. Recall that torch *accumulates* gradients. Before passing in a
        # new instance, you need to zero out the gradients from the old
        # instance
        model.zero_grad()

        # Step 3. Run the forward pass, getting log probabilities over next
        # words
        log_probs = model(context_idxs)

        # Step 4. Compute your loss function. (Again, Torch wants the target
        # word wrapped in a tensor)
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))

        # Step 5. Do the backward pass and update the gradient
        loss.backward()
        optimizer.step()

        # Get the Python number from a 1-element Tensor by calling tensor.item()
        total_loss += loss.item()
    print("Loss in Epoch {ep}: {l}".format(ep=epoch, l=np.round(total_loss, 2))) # The loss decreased every iteration over the training data!
    losses.append(total_loss)

[(['When', 'forty'], 'winters'), (['forty', 'winters'], 'shall'), (['winters', 'shall'], 'besiege')]
Loss in Epoch 0: 521.53
Loss in Epoch 1: 519.02
Loss in Epoch 2: 516.54
Loss in Epoch 3: 514.06
Loss in Epoch 4: 511.61
Loss in Epoch 5: 509.17
Loss in Epoch 6: 506.74
Loss in Epoch 7: 504.32
Loss in Epoch 8: 501.92
Loss in Epoch 9: 499.53


# Exercise: Computing Word Embeddings: Continuous Bag-of-Words

The Continuous Bag-of-Words model (CBOW) is frequently used in NLP deep
learning. It is a model that tries to predict words given the context of
a few words before and a few words after the target word. This is
distinct from language modeling, since CBOW is not sequential and does
not have to be probabilistic. Typcially, CBOW is used to quickly train
word embeddings, and these embeddings are used to initialize the
embeddings of some more complicated model. Usually, this is referred to
as *pretraining embeddings*. It almost always helps performance a couple
of percent.

The CBOW model is as follows. Given a target word $w_i$ and an
$N$ context window on each side, $w_{i-1}, \dots, w_{i-N}$
and $w_{i+1}, \dots, w_{i+N}$, referring to all context words
collectively as $C$, CBOW tries to minimize

\begin{align}-\log p(w_i | C) = -\log \text{Softmax}(A(\sum_{w \in C} q_w) + b)\end{align}

where $q_w$ is the embedding for word $w$.


## Exercise Layout
### 1. <u>Training CBOW Embeddings</u>
1.1) Implement a CBOW Model by completing ```class CBOW(nn.Module)``` and train it on ```raw_text```.    

1.2) Load Datasets ```tripadvisor_hotel_reviews_reduced.csv``` and ```scifi_reduced.txt```.     

1.3) Decide preprocessing steps by completing the function ```def custom_preprocess()```. Describe your decisions. Note that it's your choice to create different preprocessing functions for hotel reviews and scifi datasets or use the same preprocessing function.             

1.4) Train CBOW2 with a context width of 2 (in both directions) for the Hotel Reviews dataset.   

1.5) Train CBOW5 with a context width of 5 (in both directions) for the Hotel Reviews dataset. Are predictions made by the model sensitive towards the context size?
     
1.6) Train CBOW2 with a context width of 2 (in both directions) for the Sci-Fi story dataset.  


### 2. <u>Test your Embeddings</u>
Note - Do the following for CBOW2, and optionally for CBOW5

2.1) For the hotel reviews dataset, choose 3 nouns, 3 verbs, and 3 adjectives. Make sure that some nouns/verbs/adjectives occur frequently in the corpus and that others are rare. For each of the 9 chosen words, retrieve the 5 closest words according to your trained CBOW2 model. List them in your report and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   

2.2) Do the same for Sci-Fi dataset.   

2.3) How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.   

2.4) Choose 2 words and retrieve their 5 closest neighbours according to hotel review-based embeddings and the Sci-fi-based embeddings. Do they have different neighbours? If yes, can you reason why?    

2.5) What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?   

2.6) Load the pretrained embedding model with the given code snippet and retrieve the 5 closest neighbours using the embeddings from this pretrained model for your selection of words. Compare the hereby retrieved neighbours with the ones you retrieved above. Are there any similarities? How do they differ? Can you give judgment about the embedding quality of this pretrained model?


### Tips

1. Switch from CPU to a GPU instance after you have confirmed that your training procedure is working correctly.
2. You can always save your intermediate results (embeddings, preprocessed dataset, model, etc.) in your google drive via colab



### 1.1 Create a CBOW Model by completing ```class CBOW(nn.Module)``` and test it on ```raw_text```
Implement CBOW in Pytorch by filling in the class below. Some
tips:

* Think about which parameters you need to define.
* Make sure you know what shape each operation expects. Use .view() if you need to
  reshape.

In [92]:
CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right
EMBEDDING_DIM = 20
raw_text = """We are about to study the idea of a computational process.
Computational processes are abstract beings that inhabit computers.
As they evolve, processes manipulate other abstract things called data.
The evolution of a process is directed by a pattern of rules
called a program. People create programs to direct processes. In effect,
we conjure the spirits of the computer with our spells.""".split()

# By deriving a set from `raw_text`, we deduplicate the array
vocab = set(raw_text)
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}
data = []
for i in range(2, len(raw_text) - 2):
    context = [raw_text[i - 2], raw_text[i - 1],
               raw_text[i + 1], raw_text[i + 2]]
    target = raw_text[i]
    data.append((context, target))
print(data[:20])


class CBOW(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        ### TODO
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(embedding_dim * context_size, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        ### TODO
        embeds = self.embeddings(inputs)
        context_tensor = embeds.view(embeds.size(0), -1)
        out = F.relu(self.linear1(context_tensor))
        out2 = self.linear2(out)
        output = F.log_softmax(out2, dim=1)
        return output


def make_context_vector(context, word_to_ix):
    idxs = [word_to_ix[w] for w in context]
    return torch.tensor(idxs, dtype=torch.long)


make_context_vector(data[0][0], word_to_ix)  # example

[(['We', 'are', 'to', 'study'], 'about'), (['are', 'about', 'study', 'the'], 'to'), (['about', 'to', 'the', 'idea'], 'study'), (['to', 'study', 'idea', 'of'], 'the'), (['study', 'the', 'of', 'a'], 'idea'), (['the', 'idea', 'a', 'computational'], 'of'), (['idea', 'of', 'computational', 'process.'], 'a'), (['of', 'a', 'process.', 'Computational'], 'computational'), (['a', 'computational', 'Computational', 'processes'], 'process.'), (['computational', 'process.', 'processes', 'are'], 'Computational'), (['process.', 'Computational', 'are', 'abstract'], 'processes'), (['Computational', 'processes', 'abstract', 'beings'], 'are'), (['processes', 'are', 'beings', 'that'], 'abstract'), (['are', 'abstract', 'that', 'inhabit'], 'beings'), (['abstract', 'beings', 'inhabit', 'computers.'], 'that'), (['beings', 'that', 'computers.', 'As'], 'inhabit'), (['that', 'inhabit', 'As', 'they'], 'computers.'), (['inhabit', 'computers.', 'they', 'evolve,'], 'As'), (['computers.', 'As', 'evolve,', 'processes']

tensor([24,  6, 42, 28])

In [93]:
### here are some functions to help you make the data ready for use by your model

# Function to generate batches
def generate_batches(data, batch_size=8):
    ### TODO
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]


In [94]:
# Function to get context vectors with batched data
def make_context_vector2(context_list, word_to_ix):
    ### TODO
    batch = []
    for context in context_list:
        batch.append(make_context_vector(context, word_to_ix))
    return torch.stack(batch)

# Function to get context vectors with batched data
def make_labels_idx(labels, word_to_ix):
    ### TODO
    idxs = [word_to_ix[label] for label in labels]
    return torch.tensor(idxs, dtype=torch.long)

# Function to train the model
def train(model, data, device, word_to_ix, NUM_EPOCHS=15):
    ### TODO
    loss_func = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    losses=[]
    model.to(device)

    for epoch in range(NUM_EPOCHS):
        total_loss=0
        for batch in generate_batches(data, 512):
            context = [item[0] for item in batch]
            target = [item[1] for item in batch]

            context = make_context_vector2(context, word_to_ix).to(device)
            target = make_labels_idx(target, word_to_ix).to(device)

            model.zero_grad()
            log_probs = model(context)
            loss = loss_func(log_probs, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print("Loss in Epoch {ep}: {l}".format(ep=epoch, l=np.round(total_loss, 2)))
        losses.append(total_loss)

    return losses

In [95]:
### create your model and train
    
model = CBOW(vocab_size, EMBEDDING_DIM, CONTEXT_SIZE * 2)
losses = train(model, data, device, word_to_ix, NUM_EPOCHS=100)[:5]

Loss in Epoch 0: 3.94
Loss in Epoch 1: 3.86
Loss in Epoch 2: 3.79
Loss in Epoch 3: 3.72
Loss in Epoch 4: 3.65
Loss in Epoch 5: 3.58
Loss in Epoch 6: 3.51
Loss in Epoch 7: 3.44
Loss in Epoch 8: 3.37
Loss in Epoch 9: 3.3
Loss in Epoch 10: 3.23
Loss in Epoch 11: 3.16
Loss in Epoch 12: 3.1
Loss in Epoch 13: 3.03
Loss in Epoch 14: 2.96
Loss in Epoch 15: 2.89
Loss in Epoch 16: 2.82
Loss in Epoch 17: 2.75
Loss in Epoch 18: 2.67
Loss in Epoch 19: 2.6
Loss in Epoch 20: 2.53
Loss in Epoch 21: 2.46
Loss in Epoch 22: 2.38
Loss in Epoch 23: 2.31
Loss in Epoch 24: 2.24
Loss in Epoch 25: 2.16
Loss in Epoch 26: 2.08
Loss in Epoch 27: 2.01
Loss in Epoch 28: 1.93
Loss in Epoch 29: 1.86
Loss in Epoch 30: 1.78
Loss in Epoch 31: 1.71
Loss in Epoch 32: 1.63
Loss in Epoch 33: 1.55
Loss in Epoch 34: 1.48
Loss in Epoch 35: 1.41
Loss in Epoch 36: 1.33
Loss in Epoch 37: 1.26
Loss in Epoch 38: 1.19
Loss in Epoch 39: 1.13
Loss in Epoch 40: 1.06
Loss in Epoch 41: 0.99
Loss in Epoch 42: 0.93
Loss in Epoch 43: 0.87
L

### 1.2 Load Datasets

In [96]:
import pandas as pd

In [97]:
# # Download Datasets tripadvisor_hotel_reviews_reduced.csv and scifi_reduced.txt
# !mkdir "content"
# !gdown "https://drive.google.com/uc?id=1foE1JuZJeu5E_4qVge9kExzhvF32teuF" -O "content/tripadvisor_hotel_reviews_reduced.csv" # For Hotel Reviews
# !gdown "https://drive.google.com/uc?id=13IWXrTjGTrfCd9l7dScZVO8ZvMicPU75" -O "content/scifi_reduced.txt"  # For Scifi-Text

In [98]:
# local run：
!mkdir -p data
!gdown "https://drive.google.com/uc?id=1foE1JuZJeu5E_4qVge9kExzhvF32teuF" -O "data/tripadvisor_hotel_reviews_reduced.csv"
!gdown "https://drive.google.com/uc?id=13IWXrTjGTrfCd9l7dScZVO8ZvMicPU75" -O "data/scifi_reduced.txt"

A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file data already exists.
Error occurred while processing: data.
'gdown' is not recognized as an internal or external command,
operable program or batch file.
'gdown' is not recognized as an internal or external command,
operable program or batch file.


In [99]:
# ### TODO: Load the datasets
# hotel_df = pd.read_csv("content/tripadvisor_hotel_reviews_reduced.csv")
# print(f"Hotel reviews shape: {hotel_df.shape}")
# print(hotel_df.head())

# with open("content/scifi_reduced.txt", 'r', encoding='utf-8') as f:
#     scifi_text = f.read()
# print(f"Scifi text length: {len(scifi_text)} characters")

In [100]:
hotel_df = pd.read_csv("data/tripadvisor_hotel_reviews_reduced.csv")
print(f"Hotel reviews shape: {hotel_df.shape}")
print(hotel_df.head())
with open("data/scifi_reduced.txt", 'r', encoding='utf-8') as f:
    scifi_text = f.read()
print(f"Scifi text length: {len(scifi_text)} characters")

Hotel reviews shape: (10000, 2)
                                              Review  Rating
0  fantastic service large hotel caters business ...       5
1  great hotel modern hotel good location, locate...       4
2  3 star plus glasgowjust got 30th november 4 da...       4
3  nice stayed hotel nov 19-23. great little bout...       4
4  great place wonderful hotel ideally located me...       5
Scifi text length: 43062636 characters


In [101]:
hotel_df.iloc[5]['Review']

"awesome resort terrible spa took entire family resort springbreak year wonderful time, hotel beautiful, rooms large clean stylish, staff incredible goes way accomodate, kids loved lazy river pool, tried kids club day expensive worth money, niceast touches pool waiters going handing fresh strawberries ice water lemon free people pool, ate roy blue sage enjoyed, evenings included live music courtyard nice, husband tried massage spa disappointed, massages regular basis told spa best, proved wrong, thing rushed lady did n't like experienced, friend got mani/pedicure rushed experience, know overbook appointments quite bit maybe concentrate quality quantity, overall loved resort skip pricey sub-standard spa,  "

### 1.3 Preprocess Datasets
### 🗒❓ Describe your decisions for preprocessing the datasets

In [102]:
### Import libraries for preprocessing
import re
import pandas as pd
import json
from collections import Counter

In [103]:
### Complete the preprocessing function and apply it to the datasets
def custom_preprocess_general(text, min_freq=2):
    ### TODO
    text = text.lower()
    
    # Replace - _ / with space
    text = re.sub(r'[-_/]', ' ', text)
    # Only reserve space and alphabet
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    processed_list = text.split()
    # Remove single-character words
    processed_list = [w for w in processed_list if len(w) > 1]
    # Remove low-frequency words
    freq = Counter(processed_list)
    processed_list = [w for w in processed_list if freq[w] >= min_freq]

    return processed_list

def custom_preprocess_hotel(data):
    hotel_reviews_text = ' '.join(data['Review'].tolist())

    processed_list = custom_preprocess_general(hotel_reviews_text)

    return processed_list

In [104]:
### Function to get vocab
# Input will be a list of lists
# Output - vocab set
def get_vocab(raw_llist):
    ### TODO
    return set(raw_llist)

### Function to get word-to-ix dictionary
def get_word2ix(vocab, save_loc=False):
    ### TODO
    word_to_ix = {word: i for i, word in enumerate(vocab)}

    if save_loc:
        with open(save_loc, 'w') as f:
            json.dump(word_to_ix, f)

    return word_to_ix


In [105]:
### Function to generate tuples of context-target
# Input = list of lists
# Output = list of tuples (list of context_words, target)
def get_data(raw_llist, context_window=5):
    ### TODO
    data = []
    for i in range(context_window, len(raw_llist)-context_window):
        context = [raw_llist[i+j] for j in range(-context_window, context_window+1) if j != 0]
        target = raw_llist[i]
        data.append((context, target))
    return data


### 1.4 Train CBOW2 with a context width of 2 (in both directions) for the Hotel Reviews dataset.

In [106]:
EMBEDDING_DIM = 100
NUM_EPOCHS=50
CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right
# Preprocess data
hotel_list = custom_preprocess_hotel(hotel_df)

vocab_hotel = get_vocab(hotel_list)
word_to_ix_hotel = get_word2ix(vocab_hotel)
data_hotel = get_data(hotel_list, 2)


hotel_cbow2_model = CBOW(len(vocab_hotel), EMBEDDING_DIM, CONTEXT_SIZE * 2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
losses_hotel_cbow2 = train(hotel_cbow2_model, data_hotel, device, word_to_ix_hotel, NUM_EPOCHS=NUM_EPOCHS)

Loss in Epoch 0: 14265.24
Loss in Epoch 1: 12775.29
Loss in Epoch 2: 11936.0
Loss in Epoch 3: 11313.17
Loss in Epoch 4: 10823.29
Loss in Epoch 5: 10450.73
Loss in Epoch 6: 10185.59
Loss in Epoch 7: 9989.08
Loss in Epoch 8: 9828.6
Loss in Epoch 9: 9689.45
Loss in Epoch 10: 9564.52
Loss in Epoch 11: 9450.54
Loss in Epoch 12: 9345.67
Loss in Epoch 13: 9247.91
Loss in Epoch 14: 9156.08
Loss in Epoch 15: 9069.94
Loss in Epoch 16: 8988.47
Loss in Epoch 17: 8911.0
Loss in Epoch 18: 8837.54
Loss in Epoch 19: 8767.22
Loss in Epoch 20: 8700.03
Loss in Epoch 21: 8635.76
Loss in Epoch 22: 8574.07
Loss in Epoch 23: 8514.82
Loss in Epoch 24: 8457.69
Loss in Epoch 25: 8402.99
Loss in Epoch 26: 8350.01
Loss in Epoch 27: 8298.66
Loss in Epoch 28: 8249.25
Loss in Epoch 29: 8201.23
Loss in Epoch 30: 8154.74
Loss in Epoch 31: 8109.8
Loss in Epoch 32: 8066.35
Loss in Epoch 33: 8024.19
Loss in Epoch 34: 7982.91
Loss in Epoch 35: 7943.17
Loss in Epoch 36: 7904.36
Loss in Epoch 37: 7866.56
Loss in Epoch 38: 7

In [107]:
# save the model
torch.save(hotel_cbow2_model.state_dict(), "models/hotel_cbow2_model.pth")

In [109]:
print(f"Using device: {device}")

Using device: cuda


### 1.5 Train CBOW5 with a context width of 5 (in both directions) for the Hotel Reviews dataset.  

🗒❓ Are predictions made by the model sensitive towards the context size?

In [110]:
EMBEDDING_DIM = 100
NUM_EPOCHS=50
CONTEXT_SIZE = 5  # 2 words to the left, 2 to the right
# Preprocess data (reuse hotel data from 1.4)
hotel_list = custom_preprocess_hotel(hotel_df)

vocab_hotel = get_vocab(hotel_list)
word_to_ix_hotel = get_word2ix(vocab_hotel)
data_hotel_cbow5 = get_data(hotel_list, 5)


hotel_cbow5_model = CBOW(len(vocab_hotel), EMBEDDING_DIM, CONTEXT_SIZE * 2)

losses_hotel_cbow5 = train(hotel_cbow5_model, data_hotel_cbow5, device, word_to_ix_hotel, NUM_EPOCHS=NUM_EPOCHS)

Loss in Epoch 0: 14339.69
Loss in Epoch 1: 12816.24
Loss in Epoch 2: 11944.93
Loss in Epoch 3: 11294.44
Loss in Epoch 4: 10777.51
Loss in Epoch 5: 10379.44
Loss in Epoch 6: 10092.99
Loss in Epoch 7: 9881.92
Loss in Epoch 8: 9709.62
Loss in Epoch 9: 9560.72
Loss in Epoch 10: 9426.77
Loss in Epoch 11: 9304.7
Loss in Epoch 12: 9191.4
Loss in Epoch 13: 9085.86
Loss in Epoch 14: 8986.45
Loss in Epoch 15: 8892.8
Loss in Epoch 16: 8803.32
Loss in Epoch 17: 8718.13
Loss in Epoch 18: 8637.81
Loss in Epoch 19: 8559.88
Loss in Epoch 20: 8485.81
Loss in Epoch 21: 8413.49
Loss in Epoch 22: 8344.94
Loss in Epoch 23: 8278.59
Loss in Epoch 24: 8214.96
Loss in Epoch 25: 8152.82
Loss in Epoch 26: 8092.37
Loss in Epoch 27: 8034.08
Loss in Epoch 28: 7978.08
Loss in Epoch 29: 7923.15
Loss in Epoch 30: 7870.66
Loss in Epoch 31: 7819.18
Loss in Epoch 32: 7768.86
Loss in Epoch 33: 7720.56
Loss in Epoch 34: 7673.63
Loss in Epoch 35: 7626.84
Loss in Epoch 36: 7583.03
Loss in Epoch 37: 7539.35
Loss in Epoch 38: 

In [111]:
# save the model
torch.save(hotel_cbow5_model.state_dict(), "models/hotel_cbow5_model.pth")

### 1.6 Train CBOW2 with a context width of 2 (in both directions) for the Sci-Fi story dataset

In [112]:
EMBEDDING_DIM = 100
NUM_EPOCHS=50
CONTEXT_SIZE = 2  # 2 words to the left, 2 to the right
# Preprocess data
scifi_list = custom_preprocess_general(scifi_text)

vocab_scifi = get_vocab(scifi_list)
word_to_ix_scifi = get_word2ix(vocab_scifi)
data_scifi = get_data(scifi_list, 2)


scifi_cbow2_model = CBOW(len(vocab_scifi), EMBEDDING_DIM, CONTEXT_SIZE * 2)

losses_scifi_cbow2 = train(scifi_cbow2_model, data_scifi, device, word_to_ix_scifi, NUM_EPOCHS=NUM_EPOCHS)

Loss in Epoch 0: 88205.8
Loss in Epoch 1: 80230.79
Loss in Epoch 2: 78189.57
Loss in Epoch 3: 77354.78
Loss in Epoch 4: 76940.66
Loss in Epoch 5: 76718.64
Loss in Epoch 6: 76595.86
Loss in Epoch 7: 76525.64
Loss in Epoch 8: 76483.9
Loss in Epoch 9: 76465.01
Loss in Epoch 10: 76470.03
Loss in Epoch 11: 76478.37
Loss in Epoch 12: 76498.7
Loss in Epoch 13: 76526.74
Loss in Epoch 14: 76560.3
Loss in Epoch 15: 76595.8
Loss in Epoch 16: 76641.14
Loss in Epoch 17: 76689.21
Loss in Epoch 18: 76737.23
Loss in Epoch 19: 76789.15
Loss in Epoch 20: 76853.47
Loss in Epoch 21: 76906.57
Loss in Epoch 22: 76973.66
Loss in Epoch 23: 77034.6
Loss in Epoch 24: 77108.32
Loss in Epoch 25: 77173.93
Loss in Epoch 26: 77243.4
Loss in Epoch 27: 77322.02
Loss in Epoch 28: 77389.77
Loss in Epoch 29: 77462.46
Loss in Epoch 30: 77547.56
Loss in Epoch 31: 77619.84
Loss in Epoch 32: 77703.59
Loss in Epoch 33: 77790.03
Loss in Epoch 34: 77868.53
Loss in Epoch 35: 77953.44
Loss in Epoch 36: 78030.0
Loss in Epoch 37: 7

In [113]:
# save the model
torch.save(scifi_cbow2_model.state_dict(), "models/scifi_cbow2_model.pth")

#### Optional: skip training and load models from "models/file.pth"

In [114]:
# same as training
# EMBEDDING_DIM = 100
# CONTEXT_SIZE = 2
# # Load hotel_cbow2_model model

# import pandas as pd
# hotel_df = pd.read_csv("data/tripadvisor_hotel_reviews_reduced.csv")
# hotel_list = custom_preprocess_hotel(hotel_df)
# vocab_hotel = get_vocab(hotel_list)
# word_to_ix_hotel = get_word2ix(vocab_hotel)


# hotel_cbow2_model = CBOW(len(vocab_hotel), EMBEDDING_DIM, 2 * CONTEXT_SIZE)


# hotel_cbow2_model.load_state_dict(torch.load("models/hotel_cbow2_model.pth"))


# hotel_cbow2_model.to(device)
# hotel_cbow2_model.eval()

In [115]:
# EMBEDDING_DIM = 100
# CONTEXT_SIZE = 5
# # Load hotel_cbow5_model model
# hotel_cbow5_model = CBOW(len(vocab_hotel), EMBEDDING_DIM, 2 * CONTEXT_SIZE)
# hotel_cbow5_model.load_state_dict(torch.load("models/hotel_cbow5_model.pth"))
# hotel_cbow5_model.to(device)
# hotel_cbow5_model.eval()

In [116]:
# EMBEDDING_DIM = 100
# CONTEXT_SIZE = 2
# # Load scifi_cbow2_model model
# with open("data/scifi_reduced.txt", 'r', encoding='utf-8') as f:
#     scifi_text = f.read()
# scifi_list = custom_preprocess_general(scifi_text)
# vocab_scifi = get_vocab(scifi_list)
# word_to_ix_scifi = get_word2ix(vocab_scifi)

# scifi_cbow2_model = CBOW(len(vocab_scifi), EMBEDDING_DIM, 2 * CONTEXT_SIZE)
# scifi_cbow2_model.load_state_dict(torch.load("models/scifi_cbow2_model.pth"))
# scifi_cbow2_model.to(device)
# scifi_cbow2_model.eval()

### 2.1 For the hotel reviews dataset, choose 3 nouns, 3 verbs, and 3 adjectives. (CBOW2 and optionally for CBOW5)
Make sure that some nouns/verbs/adjectives occur frequently in the corpus and that others are rare. For each of the 9 chosen words, retrieve the 5 closest words according to your trained CBOW2 model.    

🗒❓ List them in your report (at the end of this notebook) and comment on the performance of your model: do the neighbours the model provides make sense? Discuss.   


In [117]:
final_9_words = [
    # Nouns
    'hotel',      # freq: 24085
    'room',       # freq: 16602
    'breakfast',  # freq: 4546
    
    # Verbs
    'stayed',     # freq: 5134
    'like',       # freq: 3988
    'complained', # freq: 175 (low-freq)
    
    # Adjectives
    'great',      # freq: 10272
    'nice',       # freq: 6119
    'luxurious',  # freq: 198 (low-freq)
]

In [118]:
import torch
import torch.nn.functional as F

def get_closest_words(word, model, word_to_ix, ix_to_word, top_k=5):
    """
    Find the top_k most similar words to the given word
    
    Parameters:
    - word: target word
    - model: trained CBOW model
    - word_to_ix: word to index dictionary
    - ix_to_word: index to word dictionary
    - top_k: number of most similar words to return
    """
    if word not in word_to_ix:
        return f"'{word}' not in vocabulary"
    
    # Get word index and embedding
    word_idx = word_to_ix[word]
    word_embedding = model.embeddings.weight[word_idx].unsqueeze(0)  # shape: [1, embedding_dim]
    
    # Get all embeddings
    all_embeddings = model.embeddings.weight  # shape: [vocab_size, embedding_dim]
    
    # Calculate cosine similarity
    similarities = F.cosine_similarity(word_embedding, all_embeddings, dim=1)
    
    # Get top_k+1 most similar words (+1 because it includes the word itself)
    top_similarities, top_indices = torch.topk(similarities, top_k + 1)
    
    # Build results list (exclude the word itself)
    results = []
    for idx, sim in zip(top_indices[1:], top_similarities[1:]):  # Skip first (itself)
        results.append((ix_to_word[idx.item()], sim.item()))
    
    return results

# Create ix_to_word dictionary for hotel reviews
ix_to_word_hotel = {i: word for word, i in word_to_ix_hotel.items()}

# Find the 5 most similar words for each selected word
print("=" * 70)
print("HOTEL REVIEWS - CBOW2 Model Results")
print("=" * 70)

for word in final_9_words:
    print(f"\nWord: '{word}'")
    closest = get_closest_words(word, hotel_cbow2_model, word_to_ix_hotel, ix_to_word_hotel, top_k=5)
    
    if isinstance(closest, str):  # If word not in vocabulary
        print(f"   {closest}")
    else:
        print("   Top 5 most similar words:")
        for i, (neighbor, similarity) in enumerate(closest, 1):
            print(f"   {i}. {neighbor:<15} (similarity: {similarity:.4f})")

HOTEL REVIEWS - CBOW2 Model Results

Word: 'hotel'
   Top 5 most similar words:
   1. resort          (similarity: 0.3827)
   2. hotelroom       (similarity: 0.3794)
   3. sumner          (similarity: 0.3544)
   4. touchi          (similarity: 0.3400)
   5. slabs           (similarity: 0.3377)

Word: 'room'
   Top 5 most similar words:
   1. apartment       (similarity: 0.4087)
   2. rooms           (similarity: 0.3693)
   3. upstair         (similarity: 0.3647)
   4. receptionthe    (similarity: 0.3628)
   5. prompty         (similarity: 0.3376)

Word: 'breakfast'
   Top 5 most similar words:
   1. breakfasts      (similarity: 0.5277)
   2. creak           (similarity: 0.4525)
   3. bussing         (similarity: 0.4363)
   4. quietbreakfast  (similarity: 0.4252)
   5. deli            (similarity: 0.4113)

Word: 'stayed'
   Top 5 most similar words:
   1. booked          (similarity: 0.4695)
   2. staying         (similarity: 0.3943)
   3. orto            (similarity: 0.3672)
   4. sibl

### 2.1 ANSWER


### 2.2 Repeat 2.1 for SciFi Dataset

🗒❓ List your findings for SciFi Dataset as well, similarly to 2.1

In [119]:
final_9_words_scifi = [
    # Nouns
    'ship',    # freq: 6296
    'planet',  # freq: 3348
    'robot',   # freq: 825 (low-freq)

    # Verbs
    'said',    # freq: 36704
    'looked',  # freq: 8489
    'teleported', # freq: 36 (low-freq)

    # Adjectives
    'little',  # freq: 8525
    'great',   # freq: 4255
    'cosmic',  # freq: 154 (low-freq)
]

In [120]:
# Create ix_to_word dictionary for scifi
ix_to_word_scifi = {i: word for word, i in word_to_ix_scifi.items()}

# Find the 5 most similar words for each selected scifi word
print("=" * 70)
print("SCI-FI - CBOW2 Model Results")
print("=" * 70)

for word in final_9_words_scifi:
    print(f"\nWord: '{word}'")
    closest = get_closest_words(word, scifi_cbow2_model, word_to_ix_scifi, ix_to_word_scifi, top_k=5)
    
    if isinstance(closest, str):  # If word not in vocabulary
        print(f"   {closest}")
    else:
        print("   Top 5 most similar words:")
        for i, (neighbor, similarity) in enumerate(closest, 1):
            print(f"   {i}. {neighbor:<15} (similarity: {similarity:.4f})")

SCI-FI - CBOW2 Model Results

Word: 'ship'
   Top 5 most similar words:
   1. machine         (similarity: 0.8547)
   2. house           (similarity: 0.8262)
   3. car             (similarity: 0.8082)
   4. room            (similarity: 0.8051)
   5. moon            (similarity: 0.7664)

Word: 'planet'
   Top 5 most similar words:
   1. world           (similarity: 0.8655)
   2. place           (similarity: 0.8444)
   3. trip            (similarity: 0.7482)
   4. universe        (similarity: 0.7460)
   5. project         (similarity: 0.7357)

Word: 'robot'
   Top 5 most similar words:
   1. ship            (similarity: 0.6475)
   2. machine         (similarity: 0.6457)
   3. moon            (similarity: 0.6439)
   4. guard           (similarity: 0.6391)
   5. ships           (similarity: 0.6343)

Word: 'said'
   Top 5 most similar words:
   1. asked           (similarity: 0.9053)
   2. nodded          (similarity: 0.8449)
   3. replied         (similarity: 0.8363)
   4. answered        

### 2.2 ANSWER


### 2.3 🗒❓ How does the quality of the hotel review-based embeddings compare with the Sci-fi-based embeddings? Elaborate.

In [121]:
# Compare average similarity scores
hotel_avg_similarities = []
scifi_avg_similarities = []

for word in final_9_words:
    closest = get_closest_words(word, hotel_cbow2_model, word_to_ix_hotel, ix_to_word_hotel, top_k=5)
    if isinstance(closest, list):
        hotel_avg_similarities.extend([sim for _, sim in closest])

for word in final_9_words_scifi:
    closest = get_closest_words(word, scifi_cbow2_model, word_to_ix_scifi, ix_to_word_scifi, top_k=5)
    if isinstance(closest, list):
        scifi_avg_similarities.extend([sim for _, sim in closest])

print(f"Hotel avg similarity: {np.mean(hotel_avg_similarities):.4f}")
print(f"Sci-fi avg similarity: {np.mean(scifi_avg_similarities):.4f}")

Hotel avg similarity: 0.3929
Sci-fi avg similarity: 0.7028


### 2.4 Choose 2 words and retrieve their 5 closest neighbours according to hotel review-based embeddings and the Sci-fi-based embeddings.

🗒❓ Do they have different neighbours? If yes, can you reason why?

In [122]:
# 2.4 Choose 2 words and compare their neighbors across hotel and scifi embeddings

# Common words likely to appear in both datasets
cross_dataset_words = ['great', 'little']

print("=" * 70)
print("2.4 Cross-Dataset Comparison")
print("=" * 70)

for word in cross_dataset_words:
    print(f"\n{'='*70}")
    print(f"Word: '{word}'")
    print(f"{'='*70}")
    
    # Hotel reviews embeddings
    print(f"\n--- HOTEL REVIEWS EMBEDDINGS ---")
    if word in word_to_ix_hotel:
        closest_hotel = get_closest_words(word, hotel_cbow2_model, word_to_ix_hotel, ix_to_word_hotel, top_k=5)
        if isinstance(closest_hotel, str):
            print(f"   {closest_hotel}")
        else:
            print("   Top 5 most similar words:")
            for i, (neighbor, similarity) in enumerate(closest_hotel, 1):
                print(f"   {i}. {neighbor:<15} (similarity: {similarity:.4f})")
    else:
        print(f"   '{word}' not in hotel reviews vocabulary")
    
    # Sci-Fi embeddings
    print(f"\n--- SCI-FI EMBEDDINGS ---")
    if word in word_to_ix_scifi:
        closest_scifi = get_closest_words(word, scifi_cbow2_model, word_to_ix_scifi, ix_to_word_scifi, top_k=5)
        if isinstance(closest_scifi, str):
            print(f"   {closest_scifi}")
        else:
            print("   Top 5 most similar words:")
            for i, (neighbor, similarity) in enumerate(closest_scifi, 1):
                print(f"   {i}. {neighbor:<15} (similarity: {similarity:.4f})")
    else:
        print(f"   '{word}' not in sci-fi vocabulary")

print("\n" + "="*70)

2.4 Cross-Dataset Comparison

Word: 'great'

--- HOTEL REVIEWS EMBEDDINGS ---
   Top 5 most similar words:
   1. good            (similarity: 0.5885)
   2. wonderful       (similarity: 0.5081)
   3. fantastic       (similarity: 0.4943)
   4. awesome         (similarity: 0.4535)
   5. perfect         (similarity: 0.4462)

--- SCI-FI EMBEDDINGS ---
   Top 5 most similar words:
   1. big             (similarity: 0.8154)
   2. small           (similarity: 0.8017)
   3. simple          (similarity: 0.7754)
   4. real            (similarity: 0.7592)
   5. heavy           (similarity: 0.7561)

Word: 'little'

--- HOTEL REVIEWS EMBEDDINGS ---
   Top 5 most similar words:
   1. bit             (similarity: 0.4938)
   2. fera            (similarity: 0.4217)
   3. courtyardi      (similarity: 0.4192)
   4. slightly        (similarity: 0.3695)
   5. cheking         (similarity: 0.3615)

--- SCI-FI EMBEDDINGS ---
   Top 5 most similar words:
   1. big             (similarity: 0.7280)
   2. hard    

### 2.4 ANSWER


### 2.5 🗒❓ What are the differences between CBOW2 and CBOW5 ? Can you "describe" them?    

In [123]:
# 2.5 Compare CBOW2 and CBOW5 for hotel reviews

comparison_words = ['hotel', 'room', 'great']

print("=" * 70)
print("2.5 CBOW2 vs CBOW5 Comparison (Hotel Reviews)")
print("=" * 70)

for word in comparison_words:
    print(f"\n{'='*70}")
    print(f"Word: '{word}'")
    print(f"{'='*70}")
    
    # CBOW2 results
    print(f"\n--- CBOW2 (context window = 2) ---")
    if word in word_to_ix_hotel:
        closest_cbow2 = get_closest_words(word, hotel_cbow2_model, word_to_ix_hotel, ix_to_word_hotel, top_k=5)
        if isinstance(closest_cbow2, str):
            print(f"   {closest_cbow2}")
        else:
            print("   Top 5 most similar words:")
            for i, (neighbor, similarity) in enumerate(closest_cbow2, 1):
                print(f"   {i}. {neighbor:<15} (similarity: {similarity:.4f})")
    else:
        print(f"   '{word}' not in vocabulary")
    
    # CBOW5 results
    print(f"\n--- CBOW5 (context window = 5) ---")
    if word in word_to_ix_hotel:
        closest_cbow5 = get_closest_words(word, hotel_cbow5_model, word_to_ix_hotel, ix_to_word_hotel, top_k=5)
        if isinstance(closest_cbow5, str):
            print(f"   {closest_cbow5}")
        else:
            print("   Top 5 most similar words:")
            for i, (neighbor, similarity) in enumerate(closest_cbow5, 1):
                print(f"   {i}. {neighbor:<15} (similarity: {similarity:.4f})")
    else:
        print(f"   '{word}' not in vocabulary")

print("\n" + "="*70)

2.5 CBOW2 vs CBOW5 Comparison (Hotel Reviews)

Word: 'hotel'

--- CBOW2 (context window = 2) ---
   Top 5 most similar words:
   1. resort          (similarity: 0.3827)
   2. hotelroom       (similarity: 0.3794)
   3. sumner          (similarity: 0.3544)
   4. touchi          (similarity: 0.3400)
   5. slabs           (similarity: 0.3377)

--- CBOW5 (context window = 5) ---
   Top 5 most similar words:
   1. apathetic       (similarity: 0.4049)
   2. establishment   (similarity: 0.3856)
   3. cleanbeach      (similarity: 0.3663)
   4. hotelall        (similarity: 0.3651)
   5. property        (similarity: 0.3602)

Word: 'room'

--- CBOW2 (context window = 2) ---
   Top 5 most similar words:
   1. apartment       (similarity: 0.4087)
   2. rooms           (similarity: 0.3693)
   3. upstair         (similarity: 0.3647)
   4. receptionthe    (similarity: 0.3628)
   5. prompty         (similarity: 0.3376)

--- CBOW5 (context window = 5) ---
   Top 5 most similar words:
   1. rooms         

### 2.5 ANSWER



### 2.6 Load the pretrained embedding model with the given code snippet and retrieve the 5 closest neighbours using the embeddings from this pretrained model for your selection of words. Compare the hereby retrieved neighbours with the ones you retrieved above. Are there any similarities? How do they differ? Can you give judgment about the embedding quality of this pretrained model?

In [124]:
# Load Pretrained GloVe Embeddings
# Install required packages
!pip install gensim
!pip install "numpy<=1.26.0"

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.0-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.0-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.2
    Uninstalling numpy-2.1.2:
      Successfully uninstalled numpy-2.1.2


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.


In [125]:
import numpy as np
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

ModuleNotFoundError: No module named 'numpy.char'

In [ ]:
# Download GloVe pretrained embeddings
!wget http://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip
!unzip -q glove.6B.zip -d glove/

In [ ]:
# Choose model
glove_input_file = "glove/glove.6B.100d.txt"
word2vec_output_file = "glove/glove.6B.100d.word2vec.txt"

# Convert GloVe format → Word2Vec format
glove2word2vec(glove_input_file, word2vec_output_file)

# Load model (this may take ~1–2 min)
glove_model = KeyedVectors.load_word2vec_format(word2vec_output_file, binary=False)
print(f"Loaded {len(glove_model.key_to_index):,} word vectors.")

In [ ]:
# 2.6 Compare with pretrained GloVe embeddings

trip_nouns = ['hotel', 'room', 'breakfast']
trip_verbs = ['stayed', 'like', 'complained']
trip_adj = ['great', 'nice', 'luxurious']

print("=" * 70)
print("2.6 Comparison with Pretrained GloVe Embeddings")
print("=" * 70)

print("\n" + "="*70)
print("HOTEL REVIEWS VOCABULARY - GloVe Results")
print("="*70)

for category, words_list in [("NOUNS", trip_nouns), ("VERBS", trip_verbs), ("ADJECTIVES", trip_adj)]:
    print(f"\n--- {category} ---")
    for w in words_list:
        if w in glove_model.key_to_index:
            print(f"\nWord: '{w}'")
            print("   Top 5 GloVe neighbors:")
            for neighbor, sim in glove_model.most_similar(w, topn=5):
                print(f"      {neighbor:<15} (cosine similarity = {sim:.3f})")
        else:
            print(f"\n'{w}' not in GloVe vocabulary.")

### 3. Use the following code snippet to load a pretrained GloVe embedding model
GloVe (Global Vectors for Word Representation) (link to paper: https://aclanthology.org/D14-1162/) is a count-based model trained on very large text corpora (Wikipedia, Common Crawl, Twitter, etc.). Unlike CBOW, which learns to predict a target word from its local context, GloVe learns embeddings that capture global co-occurrence statistics of words across the entire corpus.

Each word in GloVe has one static vector, i.e. its embedding does not change depending on context.

Note: Loading pretrained GloVe embeddings does not require a GPU. This runs efficiently on CPU. Change your run time to CPU again to save GPU compute units.

In [1]:
!pip install gensim
!pip install "numpy<=1.26.0"

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


In [2]:
import numpy as np
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

In [3]:
!wget http://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip
!unzip -q glove.6B.zip -d glove/

--2025-10-25 14:45:41--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-10-25 14:45:41--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-10-25 14:45:42--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [4]:
# Choose model
glove_input_file = "glove/glove.6B.100d.txt"
word2vec_output_file = "glove/glove.6B.100d.word2vec.txt"

# Convert GloVe format → Word2Vec format
glove2word2vec(glove_input_file, word2vec_output_file)

# Load model (this may take ~1–2 min)
model = KeyedVectors.load_word2vec_format(word2vec_output_file, binary=False)
print(f"Loaded {len(model.key_to_index):,} word vectors.")

/var/folders/tj/mf5_4wtn5nqb91zbpn4ctkkh0000gn/T/ipykernel_46139/2719571235.py:6: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, word2vec_output_file)


Loaded 400,000 word vectors.


In [5]:
### Insert your word selection from above here
trip_nouns = ['hotel','room','breakfast',]
trip_verbs = ['stayed','like','complained',]
trip_adj = ['great','nice','luxurious',]

for words_to_check in [trip_nouns, trip_verbs, trip_adj]:
  for w in words_to_check:
      if w in model.key_to_index:
          print(f"\nNearest neighbours for '{w}' in pretrained GloVe:")
          for neighbor, sim in model.most_similar(w, topn=5):
              print(f"  {neighbor:>12s}   (cosine similarity = {sim:.3f})")
      else:
          print(f"\n'{w}' not in vocabulary.")


Nearest neighbours for 'hotel' in pretrained GloVe:
        hotels   (cosine similarity = 0.793)
    restaurant   (cosine similarity = 0.776)
     apartment   (cosine similarity = 0.736)
           inn   (cosine similarity = 0.728)
        resort   (cosine similarity = 0.714)

Nearest neighbours for 'room' in pretrained GloVe:
         rooms   (cosine similarity = 0.842)
         floor   (cosine similarity = 0.800)
       bedroom   (cosine similarity = 0.766)
          door   (cosine similarity = 0.754)
        inside   (cosine similarity = 0.750)

Nearest neighbours for 'breakfast' in pretrained GloVe:
        dinner   (cosine similarity = 0.842)
         lunch   (cosine similarity = 0.824)
        buffet   (cosine similarity = 0.756)
         meals   (cosine similarity = 0.753)
          meal   (cosine similarity = 0.739)

Nearest neighbours for 'stayed' in pretrained GloVe:
      remained   (cosine similarity = 0.796)
       staying   (cosine similarity = 0.790)
          kept   (c

### 3.1 🗒❓ Compare the hereby retrieved neighbours with the ones you retrieved above. Are there any similarities to the results above? How do they differ?


### What each model returned

**GloVe (pretrained, 6B‑100d)**

* **hotel** → *hotels* (0.793), *restaurant* (0.776), *apartment* (0.736), *inn* (0.728), *resort* (0.714)
* **room** → *rooms* (0.842), *floor* (0.800), *bedroom* (0.766), *door* (0.754), *inside* (0.750)
* **breakfast** → *dinner* (0.842), *lunch* (0.824), *buffet* (0.756), *meals* (0.753), *meal* (0.739)
* (Others, e.g. adjectives, look clean too: *spacious, luxury, elegant, sumptuous*.)

**CBOW2 (self‑trained on hotel reviews)**

* **hotel** → *resort* (0.3827), *hotelroom* (0.3794), *sumner* (0.3544), *touchi* (0.3400), *slabs* (0.3377)
* **room** → *apartment* (0.4087), *rooms* (0.3693), *upstair* (0.3647), *receptionthe* (0.3628), *prompty* (0.3376)
* **breakfast** → *breakfasts* (0.5277), *creak* (0.4525), *bussing* (0.4363), …
* (For adjectives in your excerpt, items like *trendy, unending, greyline, whirlpool* show up.)

---

### Similarities

* **There is some semantic overlap.**

  * *hotel*: both models have **resort**.
  * *room*: both have **rooms** (a morphological variant) and CBOW2’s **apartment** is at least in the same “space/accommodation” theme.
  * *breakfast*: CBOW2’s **breakfasts** is the inflectional form of the GloVe query; both models keep meal‑related words in the same neighbourhood.

* **Morphology behaves sensibly in both spaces.**
  Plurals and inflections tend to cluster (e.g., *room ↔ rooms*, *breakfast ↔ breakfasts*).

---

### Differences

1. **Neighbour quality (clean vs. noisy).**

   * **GloVe** returns clean, mainstream vocabulary with obvious semantic ties: *hotel → hotels/inn/resort; room → rooms/bedroom/floor; breakfast → lunch/dinner/buffet*.
   * **CBOW2** returns many **artefacts and rare/garbled tokens**: *hotel → sumner, touchi, slabs*; *room → receptionthe, prompty, upstair*; *breakfast → creak, bussing*; adjectives mix in brand‑ish or off‑topic words like *greyline* or *whirlpool*. These look like tokenization errors, misspellings, or extremely low‑frequency items rather than genuine semantic neighbours.

2. **Similarity magnitudes.**

   * **GloVe** cosine scores sit high and tight (≈ **0.74–0.84**), suggesting a well‑structured space.
   * **CBOW2** scores are much lower and more spread (≈ **0.33–0.53**), a classic sign of small/noisy data and a less stable geometry.

3. **Type of relation emphasised.**

   * **GloVe** leans toward **class/peer similarity** (co‑hyponyms and topical peers).
   * **CBOW2** *should* lean toward **collocations** in hotel reviews, but noise dominates the top‑5 lists, masking the true domain associations (e.g., *hotel ↔ lobby/staff/check‑in*; *breakfast ↔ buffet/continental/included*).

4. **Coverage and sense bias.**

   * **GloVe** (≈400k vocab) has broad coverage and better sense averaging from diverse corpora.
   * **CBOW2** is domain‑biased (good in principle), but the signal is diluted by rare tokens and preprocessing issues, so many neighbours are not semantically meaningful.

**Quick overlap score (top‑5):**

* *hotel*: 1/5 overlap (**resort**).
* *room*: 1/5 overlap (**rooms**).
* *breakfast*: effectively 0/5 (only the plural **breakfasts** is close in form, not in GloVe’s top‑5).

---

### Why this happens

* **Data scale & diversity:** GloVe is trained on massive, heterogeneous corpora; CBOW2 used a much smaller, single‑domain dataset.
* **Preprocessing:** CBOW2 neighbours expose **tokenization & normalization problems** (e.g., *receptionthe*, *hotelroom*, misspellings like *prompty*, *upstair*).
* **Vocabulary pruning:** If `min_count` is low, junk tokens survive and crowd the neighbourhoods.
* **Model/params on small corpora:** CBOW/Skip‑gram without subword support is brittle to typos; suboptimal `window/negative/sample/epochs` can amplify noise.

---

### Conclusion

* **GloVe** gives **clean, general‑semantic** neighbours with high, consistent similarity scores—great as a robust baseline or for broad lexical similarity.
* **CBOW2 (current)** reflects the **hotel domain** but is **noisy and under‑cleaned**; its top‑5 lists are dominated by artefacts rather than true semantic/collocational neighbours.
* After stronger preprocessing, vocabulary pruning, subword modeling, and some parameter tuning, the **domain model should surface the useful collocations** (e.g., *hotel ↔ lobby/staff/check‑in; breakfast ↔ buffet/continental/included*) that GloVe underemphasizes—making it more valuable for hotel‑review tasks like aspect extraction and fine‑grained sentiment.


### Report
The lab report should contain a detailed description of the approaches you have used to solve this exercise. Please also include results.

Answers for the questions marked 🗒❓ goes here as well